In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split
import math
import random
from random import uniform

In [2]:
mnist = pd.read_csv('cp_sample.csv', sep=';')

In [ ]:
# Same split as last TP: 70% train, 15% validation, 15% test
data_train, data_test = train_test_split(mnist, test_size=0.3, random_state=4)
data_valid, data_test = train_test_split(data_test, test_size=0.5, random_state=4)
print("train:", data_train.shape, "val:", data_valid.shape, "test:", data_test.shape)

In [3]:
from sklearn.neighbors import KNeighborsClassifier

In [4]:
n_n = KNeighborsClassifier(n_neighbors=1) # this initializes a knn for k = 1
n_n.fit(data_train.iloc[:,1:],data_train.label)
# here we fit this knn by giving the features of all the
# examples of the training set (columns 1 to last) and their labels

NameError: name 'data_train' is not defined

In [ ]:
pred = n_n.predict(data_train.iloc[:2, 1:])
print("Predictions:", pred)
print("True labels:", data_train.label.iloc[:2].tolist())
# k=1 on the training set: each point is its own nearest neighbour, so both predictions are correct.

In [ ]:
print("Validation accuracy (k=1):", n_n.score(data_valid.iloc[:, 1:], data_valid.label))

In [ ]:
# Select k on the validation set (train is optimistic, especially for k=1).
results = []
saved = {}
for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(data_train.iloc[:, 1:], data_train.label)
    acc_tr = knn.score(data_train.iloc[:, 1:], data_train.label)
    acc_va = knn.score(data_valid.iloc[:, 1:], data_valid.label)
    print(f"k={k:2d}  train={acc_tr:.4f}  val={acc_va:.4f}")
    results.append((k, acc_va))
    saved[k] = knn

best_k = max(results, key=lambda r: (r[1], -r[0]))[0]
print("Selected k (best validation accuracy):", best_k)

In [ ]:
gen_acc = saved[best_k].score(data_test.iloc[:, 1:], data_test.label)
print("Test accuracy:", gen_acc)
print("Estimated generalization error:", 1 - gen_acc)

In [ ]:
from skimage.feature import hog

def my_hog(row, ori, cell):
    img = row.iloc[1:].to_numpy().reshape(28, 28)
    return pd.Series(
        hog(
            img,
            orientations=ori,
            pixels_per_cell=(cell, cell),
            cells_per_block=(1, 1),
        )
    )

In [ ]:
hog_train = data_train.apply(my_hog, axis=1, args=(8, 14))
hog_valid = data_valid.apply(my_hog, axis=1, args=(8, 14))
hog_test = data_test.apply(my_hog, axis=1, args=(8, 14))
hog_train["label"] = data_train.label
hog_valid["label"] = data_valid.label
hog_test["label"] = data_test.label
print(hog_train.shape)  # 32 HOG features + label

In [ ]:
def select_knn(train, valid, test, k_values=range(1, 16)):
    results = []
    saved_k = {}
    Xtr, ytr = train.drop(columns="label"), train.label
    Xva, yva = valid.drop(columns="label"), valid.label
    Xte, yte = test.drop(columns="label"), test.label
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(Xtr, ytr)
        acc_tr = knn.score(Xtr, ytr)
        acc_va = knn.score(Xva, yva)
        print(f"k={k:2d}  train={acc_tr:.4f}  val={acc_va:.4f}")
        results.append((k, acc_va))
        saved_k[k] = knn
    best = max(results, key=lambda r: (r[1], -r[0]))[0]
    acc_te = saved_k[best].score(Xte, yte)
    print("Selected k:", best)
    print("Test accuracy:", acc_te)
    print("Estimated generalization error:", 1 - acc_te)
    return best, acc_te, saved_k[best]

print("=== HOG: 8 orientations, 14x14 cells ===")
select_knn(hog_train, hog_valid, hog_test)

In [ ]:
# 14x14 cells are too coarse (only 4 histograms of 8 bins = 32 features).
# Finer cells keep more spatial structure of the digit.
def make_hog(ori, cell):
    h_tr = data_train.apply(my_hog, axis=1, args=(ori, cell))
    h_va = data_valid.apply(my_hog, axis=1, args=(ori, cell))
    h_te = data_test.apply(my_hog, axis=1, args=(ori, cell))
    h_tr["label"] = data_train.label
    h_va["label"] = data_valid.label
    h_te["label"] = data_test.label
    return h_tr, h_va, h_te

print("=== HOG: 8 orientations, 7x7 cells ===")
hog_train7, hog_valid7, hog_test7 = make_hog(8, 7)
select_knn(hog_train7, hog_valid7, hog_test7)

print("\n=== HOG: 8 orientations, 4x4 cells ===")
hog_train4, hog_valid4, hog_test4 = make_hog(8, 4)
best_k_hog, gen_acc_hog, _ = select_knn(hog_train4, hog_valid4, hog_test4)

In [ ]:
# Conclusion:
# - Raw pixels, best k=6: val ~0.81, test ~0.83, gen. error ~0.17
# - HOG 14x14 is worse: too few blocks, digits 3/8/9 get mixed
# - HOG 4x4 with k=4 is the most adapted: val ~0.87, test ~0.84, gen. error ~0.16
# HOG helps when the cells are small enough to keep the shape of the digit.

In [ ]:
from collections import Counter

def wilson_clean(X, y, k=3):
    """Remove training points that are misclassified by their k other neighbours."""
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    neigh = knn.kneighbors(X, n_neighbors=k + 1, return_distance=False)[:, 1:]
    pred = np.array([Counter(row).most_common(1)[0][0] for row in y[neigh]])
    keep = pred == y
    return X[keep], y[keep], keep

def hart_condense(X, y, random_state=4):
    """Keep only points that 1-NN currently gets wrong (Hart's CNN)."""
    rng = np.random.default_rng(random_state)
    order = rng.permutation(len(y))
    store = [int(order[0])]
    stored = set(store)
    changed = True
    while changed:
        changed = False
        knn = KNeighborsClassifier(n_neighbors=1)
        knn.fit(X[store], y[store])
        for i in order:
            i = int(i)
            if i in stored:
                continue
            if knn.predict(X[i : i + 1])[0] != y[i]:
                store.append(i)
                stored.add(i)
                knn = KNeighborsClassifier(n_neighbors=1)
                knn.fit(X[store], y[store])
                changed = True
    return X[store], y[store], store

In [ ]:
X_tr = data_train.iloc[:, 1:].to_numpy()
y_tr = data_train.label.to_numpy()
X_va = data_valid.iloc[:, 1:].to_numpy()
y_va = data_valid.label.to_numpy()
X_te = data_test.iloc[:, 1:].to_numpy()
y_te = data_test.label.to_numpy()

print(f"Original training size: {len(y_tr)}")
base = KNeighborsClassifier(n_neighbors=1).fit(X_tr, y_tr)
print(f"1-NN original   val={base.score(X_va, y_va):.4f}  test={base.score(X_te, y_te):.4f}")

X_c, y_c, keep = wilson_clean(X_tr, y_tr, k=3)
print(f"\nAfter Wilson cleaning: {len(y_c)} kept ({100 * len(y_c) / len(y_tr):.1f}%)")
clean_knn = KNeighborsClassifier(n_neighbors=1).fit(X_c, y_c)
print(f"1-NN cleaned    val={clean_knn.score(X_va, y_va):.4f}  test={clean_knn.score(X_te, y_te):.4f}")

X_s, y_s, store = hart_condense(X_tr, y_tr)
print(f"\nAfter Hart condensing: {len(y_s)} kept ({100 * len(y_s) / len(y_tr):.1f}%)")
cnn = KNeighborsClassifier(n_neighbors=1).fit(X_s, y_s)
print(f"1-NN condensed  val={cnn.score(X_va, y_va):.4f}  test={cnn.score(X_te, y_te):.4f}")

X_cs, y_cs, _ = hart_condense(X_c, y_c)
print(f"\nClean then condense: {len(y_cs)} kept ({100 * len(y_cs) / len(y_tr):.1f}%)")
cc = KNeighborsClassifier(n_neighbors=1).fit(X_cs, y_cs)
print(f"1-NN clean+cnn  val={cc.score(X_va, y_va):.4f}  test={cc.score(X_te, y_te):.4f}")

In [ ]:
# Cleaning removes noisy / atypical training digits (points whose neighbours disagree).
# The set shrinks and 1-NN often becomes a bit more robust.
# Condensing keeps only points near the decision boundary (those 1-NN currently misses).
# The set shrinks a lot; accuracy usually drops a little but prediction is cheaper.
# Clean-then-condense is the usual order: first drop noise, then compress the rest.